In [ ]:
# Heart Disease Prediction - Supervised Learning Classification

## Objective
Build and evaluate classification models to predict whether a patient has heart disease. The target column is `heart_disease` (1 = disease present, 0 = absent).

## Dataset Columns
- age, sex, chest_pain_type, resting_bp, cholesterol, fasting_bs, resting_ecg, max_hr, exercise_angina, oldpeak, st_slope, heart_disease

---

## Task 1: Data Loading and Inspection

```python
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, classification_report

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# Load the dataset
df = pd.read_csv('q1_heart_disease.csv')

# Display shape
print("="*60)
print("DATASET INFORMATION")
print("="*60)
print(f"\nDataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")

# Display data types
print("\n" + "-"*40)
print("Data Types:")
print("-"*40)
print(df.dtypes)

# Display missing value counts
print("\n" + "-"*40)
print("Missing Value Counts:")
print("-"*40)
missing_counts = df.isnull().sum()
missing_percent = (missing_counts / len(df)) * 100
missing_df = pd.DataFrame({
    'Missing Count': missing_counts,
    'Percentage (%)': missing_percent
})
print(missing_df[missing_df['Missing Count'] > 0])

# Display first five rows
print("\n" + "-"*40)
print("First 5 Rows:")
print("-"*40)
print(df.head())

##Task 2: Exploratory Data Analysis
# Set style for better visualizations
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Create a figure with subplots
fig = plt.figure(figsize=(16, 14))

# 1. Target Class Distribution
ax1 = plt.subplot(2, 2, 1)
target_counts = df['heart_disease'].value_counts()
colors = ['#ff9999', '#66b3ff']
bars = ax1.bar(['No Disease (0)', 'Disease (1)'], target_counts.values, color=colors, edgecolor='black', linewidth=2)
ax1.set_title('Distribution of Heart Disease Target Variable', fontsize=14, fontweight='bold')
ax1.set_xlabel('Target Class', fontsize=12)
ax1.set_ylabel('Count', fontsize=12)

for bar, count in zip(bars, target_counts.values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5, 
             f'{count}\n({count/len(df)*100:.1f}%)', 
             ha='center', va='bottom', fontsize=11, fontweight='bold')
ax1.set_ylim(0, max(target_counts.values) + 50)

# 2. Age Distribution by Heart Disease
ax2 = plt.subplot(2, 2, 2)
df_no_disease = df[df['heart_disease'] == 0]
df_disease = df[df['heart_disease'] == 1]

ax2.hist(df_no_disease['age'], bins=20, alpha=0.6, label='No Disease', color='#ff9999', edgecolor='black', linewidth=1.5)
ax2.hist(df_disease['age'], bins=20, alpha=0.6, label='Disease', color='#66b3ff', edgecolor='black', linewidth=1.5)
ax2.set_title('Age Distribution by Heart Disease Status', fontsize=14, fontweight='bold')
ax2.set_xlabel('Age', fontsize=12)
ax2.set_ylabel('Frequency', fontsize=12)
ax2.legend(fontsize=11)
ax2.axvline(df_no_disease['age'].mean(), color='#ff9999', linestyle='dashed', linewidth=2, alpha=0.7)
ax2.axvline(df_disease['age'].mean(), color='#66b3ff', linestyle='dashed', linewidth=2, alpha=0.7)

# 3. Correlation Heatmap
ax3 = plt.subplot(2, 2, 3)
numerical_cols = ['age', 'sex', 'resting_bp', 'cholesterol', 'fasting_bs', 
                  'max_hr', 'exercise_angina', 'oldpeak', 'heart_disease']
corr_matrix = df[numerical_cols].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', 
            center=0, square=True, linewidths=0.5, ax=ax3, 
            cbar_kws={"shrink": 0.8, "label": "Correlation"})
ax3.set_title('Correlation Heatmap (Numerical Features)', fontsize=14, fontweight='bold')

# 4. Chest Pain Type vs Heart Disease
ax4 = plt.subplot(2, 2, 4)
chest_pain_cross = pd.crosstab(df['chest_pain_type'], df['heart_disease'], normalize='index') * 100
chest_pain_cross.plot(kind='bar', ax=ax4, color=['#ff9999', '#66b3ff'], edgecolor='black', linewidth=2)
ax4.set_title('Heart Disease Rate by Chest Pain Type', fontsize=14, fontweight='bold')
ax4.set_xlabel('Chest Pain Type', fontsize=12)
ax4.set_ylabel('Percentage (%)', fontsize=12)
ax4.legend(title='Heart Disease', labels=['No Disease', 'Disease'], fontsize=10)
ax4.set_xticklabels(ax4.get_xticklabels(), rotation=45, ha='right')

for container in ax4.containers:
    ax4.bar_label(container, fmt='%.1f%%', fontsize=9)

plt.tight_layout()
plt.show()
# Create a second figure with more visualizations
fig2, axes = plt.subplots(1, 3, figsize=(15, 5))

# 5. Sex vs Heart Disease
ax5 = axes[0]
sex_cross = pd.crosstab(df['sex'], df['heart_disease'], normalize='index') * 100
sex_cross.plot(kind='bar', ax=ax5, color=['#ff9999', '#66b3ff'], edgecolor='black')
ax5.set_title('Heart Disease Rate by Sex', fontsize=12, fontweight='bold')
ax5.set_xlabel('Sex (0=Female, 1=Male)', fontsize=10)
ax5.set_ylabel('Percentage (%)', fontsize=10)
ax5.legend(title='Heart Disease', labels=['No Disease', 'Disease'])
ax5.set_xticklabels(['Female', 'Male'], rotation=0)
for container in ax5.containers:
    ax5.bar_label(container, fmt='%.1f%%')

# 6. Exercise Angina vs Heart Disease
ax6 = axes[1]
angina_cross = pd.crosstab(df['exercise_angina'], df['heart_disease'], normalize='index') * 100
angina_cross.plot(kind='bar', ax=ax6, color=['#ff9999', '#66b3ff'], edgecolor='black')
ax6.set_title('Heart Disease Rate by Exercise Angina', fontsize=12, fontweight='bold')
ax6.set_xlabel('Exercise Angina (0=No, 1=Yes)', fontsize=10)
ax6.set_ylabel('Percentage (%)', fontsize=10)
ax6.legend(title='Heart Disease', labels=['No Disease', 'Disease'])
ax6.set_xticklabels(['No Angina', 'Angina'], rotation=0)
for container in ax6.containers:
    ax6.bar_label(container, fmt='%.1f%%')

# 7. Max Heart Rate vs Heart Disease (Boxplot)
ax7 = axes[2]
bp_data = [df[df['heart_disease'] == 0]['max_hr'], df[df['heart_disease'] == 1]['max_hr']]
bp_box = ax7.boxplot(bp_data, labels=['No Disease', 'Disease'], patch_artist=True,
                      boxprops=dict(linewidth=2), medianprops=dict(linewidth=2, color='red'))
bp_box['boxes'][0].set_facecolor('#ff9999')
bp_box['boxes'][1].set_facecolor('#66b3ff')
ax7.set_title('Max Heart Rate Distribution by Disease Status', fontsize=12, fontweight='bold')
ax7.set_ylabel('Max Heart Rate', fontsize=10)
ax7.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

##Task 3: Data Preprocessing
# Create a copy of the dataframe
df_processed = df.copy()

print("="*60)
print("DATA PREPROCESSING")
print("="*60)

# Handle missing values
print("\n" + "-"*40)
print("1. Handling Missing Values")
print("-"*40)

print("\nBefore handling missing values:")
print(f"  - Missing in resting_bp: {df_processed['resting_bp'].isnull().sum()} rows")
print(f"  - Missing in cholesterol: {df_processed['cholesterol'].isnull().sum()} rows")

# Strategy: Median imputation for numerical columns
# Justification: Both features may have outliers. Median is robust to outliers 
# and preserves central tendency. Dropping rows would lose ~6.6% of data.

median_resting_bp = df_processed['resting_bp'].median()
median_cholesterol = df_processed['cholesterol'].median()

df_processed['resting_bp'] = df_processed['resting_bp'].fillna(median_resting_bp)
df_processed['cholesterol'] = df_processed['cholesterol'].fillna(median_cholesterol)

print("\nAfter handling missing values:")
print(f"  - Missing in resting_bp: {df_processed['resting_bp'].isnull().sum()} rows")
print(f"  - Missing in cholesterol: {df_processed['cholesterol'].isnull().sum()} rows")
print(f"\n  Imputation values used:")
print(f"  - Median resting_bp: {median_resting_bp:.1f}")
print(f"  - Median cholesterol: {median_cholesterol:.1f}")

# Identify categorical columns for one-hot encoding
categorical_cols = ['chest_pain_type', 'resting_ecg', 'st_slope']

print("\n" + "-"*40)
print("2. One-Hot Encoding Categorical Variables")
print("-"*40)
print(f"\nCategorical columns to encode: {categorical_cols}")

# Display unique values before encoding
for col in categorical_cols:
    print(f"  - {col}: {df_processed[col].unique()}")

# Apply one-hot encoding
df_encoded = pd.get_dummies(df_processed, columns=categorical_cols, drop_first=False)
print(f"\nShape after one-hot encoding: {df_encoded.shape[0]} rows, {df_encoded.shape[1]} columns")
print(f"New columns created: {df_encoded.shape[1] - df_processed.shape[1]}")

# Separate features and target
X = df_encoded.drop('heart_disease', axis=1)
y = df_encoded['heart_disease']

print(f"\nFeatures (X) shape: {X.shape}")
print(f"Target (y) shape: {y.shape}")

# Scale numerical features
numerical_features = ['age', 'sex', 'resting_bp', 'cholesterol', 'fasting_bs', 
                      'max_hr', 'exercise_angina', 'oldpeak']

print("\n" + "-"*40)
print("3. Feature Scaling (StandardScaler)")
print("-"*40)

scaler = StandardScaler()
X[numerical_features] = scaler.fit_transform(X[numerical_features])

print("\nNumerical features scaled using StandardScaler")
print(f"Features scaled: {numerical_features}")
print("\nScaling statistics:")
for col in numerical_features:
    print(f"  - {col}: mean={scaler.mean_[numerical_features.index(col)]:.2f}, std={scaler.scale_[numerical_features.index(col)]:.2f}")

# Split the data
print("\n" + "-"*40)
print("4. Train-Test Split")
print("-"*40)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"\nTraining set size: {X_train.shape[0]} samples ({X_train.shape[0]/len(df)*100:.1f}%)")
print(f"Test set size: {X_test.shape[0]} samples ({X_test.shape[0]/len(df)*100:.1f}%)")
print(f"\nTraining set class distribution:")
print(f"  - Disease (1): {y_train.sum()} samples ({y_train.mean()*100:.1f}%)")
print(f"  - No Disease (0): {(len(y_train)-y_train.sum())} samples ({(1-y_train.mean())*100:.1f}%)")
print(f"\nTest set class distribution:")
print(f"  - Disease (1): {y_test.sum()} samples ({y_test.mean()*100:.1f}%)")
print(f"  - No Disease (0): {(len(y_test)-y_test.sum())} samples ({(1-y_test.mean())*100:.1f}%)")

##Task 4: Model Training
print("="*60)
print("MODEL TRAINING")
print("="*60)

# Initialize models with random_state=42
dt_model = DecisionTreeClassifier(random_state=42)
rf_model = RandomForestClassifier(random_state=42, n_estimators=100)
gb_model = GradientBoostingClassifier(random_state=42, n_estimators=100)

# Store models in dictionary
models = {
    'Decision Tree': dt_model,
    'Random Forest': rf_model,
    'Gradient Boosting': gb_model
}

# Train all models
trained_models = {}
training_results = []

print("\nTraining models...\n")

for name, model in models.items():
    # Train the model
    model.fit(X_train, y_train)
    trained_models[name] = model
    
    # Calculate training accuracy
    train_score = model.score(X_train, y_train)
    training_results.append({
        'Model': name,
        'Training Accuracy': f"{train_score*100:.2f}%"
    })
    
    print(f"✓ {name}: Trained successfully (Training Accuracy: {train_score*100:.2f}%)")

print("\n" + "-"*40)
print("Training Summary:")
print("-"*40)
training_df = pd.DataFrame(training_results)
print(training_df.to_string(index=False))

## Task 5: Model Evaluation
print("="*60)
print("MODEL EVALUATION")
print("="*60)

# Store evaluation results
evaluation_results = []

def evaluate_model(model, X_test, y_test, model_name):
    """
    Evaluate a classification model and print metrics.
    """
    y_pred = model.predict(X_test)
    
    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    
    # Metrics
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    accuracy = model.score(X_test, y_test)
    
    print(f"\n{'='*50}")
    print(f"{model_name}")
    print(f"{'='*50}")
    
    print("\nConfusion Matrix:")
    print(f"                 Predicted")
    print(f"                 No(0)    Yes(1)")
    print(f"Actual   No(0)   {cm[0,0]:3d}      {cm[0,1]:3d}")
    print(f"         Yes(1)  {cm[1,0]:3d}      {cm[1,1]:3d}")
    
    print(f"\nClassification Metrics:")
    print(f"  - Accuracy:  {accuracy*100:.2f}%")
    print(f"  - Precision: {precision*100:.2f}%")
    print(f"  - Recall:    {recall*100:.2f}%")
    print(f"  - F1-Score:  {f1*100:.2f}%")
    
    print(f"\nDetailed Classification Report:")
    print(classification_report(y_test, y_pred, target_names=['No Disease', 'Disease']))
    
    # Store results
    evaluation_results.append({
        'Model': model_name,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1,
        'TN': cm[0,0], 'FP': cm[0,1],
        'FN': cm[1,0], 'TP': cm[1,1]
    })
    
    return cm, y_pred

# Evaluate all models
for name, model in trained_models.items():
    evaluate_model(model, X_test, y_test, name)

# Create comparison table
print("\n" + "="*60)
print("MODEL COMPARISON SUMMARY")
print("="*60)

comparison_df = pd.DataFrame(evaluation_results)
comparison_df[['Accuracy', 'Precision', 'Recall', 'F1-Score']] = comparison_df[['Accuracy', 'Precision', 'Recall', 'F1-Score']].multiply(100).round(2)
comparison_df = comparison_df[['Model', 'Accuracy', 'Precision', 'Recall', 'F1-Score']]
print(comparison_df.to_string(index=False))

# Visual comparison of models
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Bar plot comparison
ax1 = axes[0]
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
x = np.arange(len(metrics))
width = 0.25
colors_bar = ['#2ecc71', '#3498db', '#e74c3c']

for i, (idx, row) in enumerate(comparison_df.iterrows()):
    values = [row[m] for m in metrics]
    ax1.bar(x + i*width, values, width, label=row['Model'], color=colors_bar[i], edgecolor='black')

ax1.set_xlabel('Metrics', fontsize=12)
ax1.set_ylabel('Score (%)', fontsize=12)
ax1.set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
ax1.set_xticks(x + width)
ax1.set_xticklabels(metrics)
ax1.legend()
ax1.set_ylim(60, 100)
ax1.grid(True, alpha=0.3, axis='y')

# 2. Confusion matrices heatmaps
ax2 = axes[1]
# Re-evaluate to get confusion matrices
for i, (name, model) in enumerate(trained_models.items()):
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    
    # Subplot for each model
    ax = plt.subplot(1, 3, i+1)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, 
                xticklabels=['No Disease', 'Disease'],
                yticklabels=['No Disease', 'Disease'],
                cbar=False, square=True)
    ax.set_title(f'{name}', fontsize=11)
    ax.set_xlabel('Predicted')
    if i == 0:
        ax.set_ylabel('Actual')

plt.suptitle('Confusion Matrices Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
Best Model Analysis
print("="*60)
print("BEST MODEL ANALYSIS")
print("="*60)

# Find best model based on F1-Score (balanced metric for classification)
best_model_idx = comparison_df['F1-Score'].idxmax()
best_model = comparison_df.loc[best_model_idx, 'Model']
best_f1 = comparison_df.loc[best_model_idx, 'F1-Score']
best_precision = comparison_df.loc[best_model_idx, 'Precision']
best_recall = comparison_df.loc[best_model_idx, 'Recall']

print(f"\n🏆 BEST MODEL: {best_model}")
print(f"\nPerformance Metrics:")
print(f"  - F1-Score:  {best_f1:.2f}%")
print(f"  - Precision: {best_precision:.2f}%")
print(f"  - Recall:    {best_recall:.2f}%")

# Feature importance for the best model (if available)
if best_model == 'Random Forest':
    best_model_obj = trained_models['Random Forest']
    feature_importance = pd.DataFrame({
        'feature': X.columns,
        'importance': best_model_obj.feature_importances_
    }).sort_values('importance', ascending=False).head(10)
    
    print(f"\nTop 10 Most Important Features ({best_model}):")
    print(feature_importance.to_string(index=False))
    
    # Plot feature importance
    plt.figure(figsize=(10, 6))
    plt.barh(feature_importance['feature'][::-1], feature_importance['importance'][::-1], color='#3498db', edgecolor='black')
    plt.xlabel('Importance', fontsize=12)
    plt.title(f'Top 10 Feature Importances - {best_model}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

